# Goal Setting and Monitoring

The Goal Setting and Monitoring pattern enables agents to pursue a defined objective through an iterative **generate → evaluate → refine** loop. The agent doesn't stop at the first output — it measures success against explicit criteria and continues until the goals are met or a maximum iteration count is reached.

## Implementation with Flyte v2 + the Agent harness

This refactor follows the docs' **"Strategy 1: wrap the built-in loop"**: a `GoalSeekingAgent` subclasses `Agent` and overrides `run`, delegating each generation to `super().run.aio(...)` and each judgement to a dedicated evaluator `Agent`. The outer generate → evaluate → refine loop lives in the subclass; the inner LLM ↔ tool turn loop is inherited.

#### LangChain vs Flyte v2 + Agent harness

| Aspect | LangChain | Flyte v2 + `Agent` harness |
|--------|-----------|----------------------------|
| **Outer loop** | Hand-written `while` over `goals_met()` | `GoalSeekingAgent.run` wrapping `super().run.aio` |
| **Generator / evaluator** | Two `ChatOpenAI` calls | Two `Agent`s (generator subclass + evaluator) |
| **LLM client** | LangChain wrapper | Harness' litellm callback |
| **Live progress** | `print()` | `flyte.report` HTML tab, updated each iteration |
| **Retries** | Manual `try/except` | `retries=3` on `@env.task` |
| **Secrets** | `.env` / `dotenv` | `flyte.Secret` injected by cluster |

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' litellm

### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [ ]:
!flyte start devbox

### 2. Store your API key

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-...

### 3. Import dependencies and configure the TaskEnvironment

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass, field
from datetime import timedelta

import flyte
import flyte.report
from flyte.ai.agents import Agent, AgentResult
from flyte.syncify import syncify

flyte.init_from_config()

_image = (
    flyte.Image.from_debian_base(name="goal-agent", python_version=(3, 12))
    .with_pip_packages("litellm")
)

goal_env = flyte.TaskEnvironment(
    name="goal_agent",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="2Gi"),
    secrets=[
        flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY"),
    ],
)

### 4. Define the result model

`GoalAgentResult` is the typed, serializable output surfaced in the Flyte UI: the final code, how many iterations it took, and whether the agent converged (met all goals) before the cap.

In [ ]:
@dataclass
class GoalAgentResult:
    """Final output of the goal-setting agent."""
    final_code: str
    iterations_used: int
    converged: bool

### 5. Subclass `Agent` to wrap the loop

The **generator** produces code; the **evaluator** judges whether the goals are met. Keeping them as separate agents (as in the LangChain example) avoids the self-serving rationalization problem — the evaluator sees only the code and goals, not the generator's reasoning.

`GoalSeekingAgent` overrides `run`: it calls `super().run.aio(...)` to generate, then the evaluator `Agent` to judge, and loops with feedback until PASS or `max_iterations`. Re-applying `@syncify` keeps the familiar `agent.run(...)` / `agent.run.aio(...)` calling convention.

In [ ]:
GENERATOR_SYSTEM = """\
You are an expert Python developer.
Generate clean, idiomatic Python code that satisfies the stated use case and goals.
Return ONLY the raw Python source — no prose, no markdown fences."""

EVALUATOR_SYSTEM = """\
You are a strict code reviewer. Evaluate the provided Python code against the stated goals.
Respond with exactly TWO lines:
Line 1: PASS or FAIL
Line 2: One sentence of specific, actionable feedback (even for PASS)."""


def _esc(text: str) -> str:
    return text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


@dataclass
class GoalSeekingAgent(Agent):
    """Wraps the built-in Agent loop with a generate -> evaluate -> refine outer loop."""
    max_iterations: int = 5
    evaluator: Agent | None = None

    @syncify
    async def run(self, message: str, history: list | None = None) -> AgentResult:
        feedback, previous = "", ""

        for i in range(self.max_iterations):
            gen_prompt = message
            if previous:
                gen_prompt += f"\n\nPrevious code (revise this):\n{previous}"
            if feedback:
                gen_prompt += f"\n\nFeedback to address:\n{feedback}"

            # Inner loop (LLM turns) is inherited from Agent.run.
            gen = await super(GoalSeekingAgent, self).run.aio(gen_prompt)
            previous = (gen.summary or "").strip()

            verdict = await self.evaluator.run.aio(
                f"{message}\n\nCode to evaluate:\n{previous}"
            )
            lines = (verdict.summary or "").strip().splitlines()
            passed = bool(lines) and lines[0].strip().upper().startswith("PASS")
            feedback = lines[1].strip() if len(lines) > 1 else ""

            await flyte.report.log.aio(
                f"<h3>Iteration {i + 1} — {'PASS' if passed else 'FAIL'}</h3>"
                f"<p><em>{_esc(feedback)}</em></p><pre>{_esc(previous)}</pre>"
            )
            await flyte.report.flush.aio()

            if passed:
                return AgentResult(summary=previous, attempts=i + 1)

        return AgentResult(
            summary=previous,
            attempts=self.max_iterations,
            error="Reached max_iterations without meeting all goals.",
        )

### 6. Define the goal-setting task

#### From hand-written loop to a subclassed Agent

The earlier version hand-wrote the loop in the task body — `@flyte.trace`-d `_generate` / `_evaluate` helpers, manual feedback threading, and a `for i in range(max_iterations)` block. That logic now lives in `GoalSeekingAgent.run`, so the task is a thin wrapper that builds the prompt and maps the `AgentResult` to a typed `GoalAgentResult`.

In [ ]:
goal_seeker = GoalSeekingAgent(
    name="code-generator",
    model="claude-haiku-4-5",
    instructions=GENERATOR_SYSTEM,
    max_iterations=5,
    evaluator=Agent(
        name="code-reviewer",
        model="claude-haiku-4-5",
        instructions=EVALUATOR_SYSTEM,
    ),
)


@goal_env.task(
    retries=3,
    timeout=timedelta(minutes=15),
    cache=flyte.Cache(behavior="disable"),
    report=True,
)
async def goal_agent(
    use_case: str,
    goals: list[str],
    max_iterations: int = 5,
) -> GoalAgentResult:
    """Goal-setting agent: generate -> evaluate -> refine until PASS or max iterations.

    The outer loop is owned by GoalSeekingAgent.run; each iteration is logged live to
    the report tab in the Flyte UI.
    """
    goal_seeker.max_iterations = max_iterations
    prompt = f"Use case: {use_case}\n\nGoals:\n" + "\n".join(f"- {g}" for g in goals)

    result: AgentResult = await goal_seeker.run.aio(prompt)
    return GoalAgentResult(
        final_code=result.summary,
        iterations_used=result.attempts,
        converged=result.error is None,
    )

### 7. Run locally

In [ ]:
USE_CASE = "Write a Python function that finds the BinaryGap of a positive integer."

GOALS = [
    "Functionally correct for all valid positive integers",
    "Handles edge cases: no gaps (return 0), single bit",
    "Includes a clear docstring with examples",
    "Raises ValueError for non-positive inputs",
    "Simple and readable — no unnecessary complexity",
]

run = flyte.run(goal_agent, use_case=USE_CASE, goals=GOALS, max_iterations=4)
run.wait()
result: GoalAgentResult = run.outputs()[0]

print(f"Converged: {result.converged}")
print(f"Iterations used: {result.iterations_used}")
print("\n" + "=" * 60)
print(result.final_code)

### Running remotely

Remote execution adds the live `report` tab to the Flyte UI, showing each iteration's generated code and evaluation in real time.

In [ ]:
run = flyte.run(goal_agent, use_case=USE_CASE, goals=GOALS, max_iterations=4)
run.wait()
result = run.outputs()[0]
print(f"Converged: {result.converged}, Iterations: {result.iterations_used}")
print(result.final_code)

## Scaling the pattern

The devbox runs each task in a fresh container. For production workloads with many short LLM calls, `ReusePolicy` eliminates cold-start overhead by keeping a pool of warm containers ready.

> **Note:** `ReusePolicy` is a Union-specific feature that requires a [Union deployment](https://www.union.ai/docs/v2/union/). It is not supported on the local devbox.

In [ ]:
# Requires a Union deployment — not supported on the local devbox
from datetime import timedelta

production_goal_agent = flyte.TaskEnvironment(
    name="goal_agent_prod",
    image=_image,
    resources=flyte.Resources(cpu="2", memory="4Gi"),
    secrets=[flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY")],
    reusable=flyte.ReusePolicy(
        replicas=(2, 8),
        concurrency=4,
        scaledown_ttl=timedelta(minutes=5),
        idle_ttl=timedelta(minutes=15),
    ),
)